In [ ]:
import pandas as pd

event = pd.read_csv('output_anomalies/RF_run_with_lagged_et01_event.csv')


poor_sites = pd.read_csv(
    "poor_performance_sites.csv"
)

event_filtered = event[~event["SITE_ID"].isin(poor_sites["SITE_ID"])]

anomalies_per_site = (
    event_filtered
    .groupby("SITE_ID")
    .size()
    .reset_index(name="num_anomalies")
)

(anomalies_per_site)

,SITE_ID,num_anomalies
0,AU-ASM,28
1,AU-Cpr,43
2,AU-Dry,30
3,AU-Gin,31
4,AU-How,13
5,AU-RDF,23
6,AU-Rig,41
7,AU-Stp,47
8,AU-TTE,30
9,AU-Wom,12


In [ ]:
import pandas as pd
import numpy as np

uncertainty = pd.read_csv('output_anomalies/RF_run_with_lagged_et01_testuncertainty.csv')

event_filtered = event_filtered.merge(
    uncertainty[["SITE_ID", "p25", "p75"]],
    on="SITE_ID",
    how="left"
)

event_filtered["outside_IQR"] = (
    (event_filtered["actual - predicted"] < event_filtered["p25"]) |
    (event_filtered["actual - predicted"] > event_filtered["p75"])
)
outside_range = (
    event_filtered
    .groupby("SITE_ID")["outside_IQR"]
    .sum()
    .reset_index(name="num_anomalies_outside_IQR")
)

outside_range

,SITE_ID,num_anomalies_outside_IQR
0,AU-ASM,28
1,AU-Cpr,28
2,AU-Dry,23
3,AU-Gin,21
4,AU-How,11
5,AU-RDF,22
6,AU-Rig,20
7,AU-Stp,45
8,AU-TTE,29
9,AU-Wom,5


In [37]:
outside_range.shape[0]

60

In [45]:
outside_range[outside_range["num_anomalies_outside_IQR"] > 4].shape[0]

48

In [54]:
comparison = anomalies_per_site.merge(
    outside_range,
    on="SITE_ID",
    how="left"
)

comparison["num_anomalies_outside_IQR"] = comparison["num_anomalies_outside_IQR"]
comparison["% of anomalies outside range"] = round(comparison["num_anomalies_outside_IQR"] / comparison["num_anomalies"] * 100, 2)

comparison

,SITE_ID,num_anomalies,num_anomalies_outside_IQR,% of anomalies outside range
0,AU-ASM,28,28,100.00
1,AU-Cpr,43,28,65.12
2,AU-Dry,30,23,76.67
3,AU-Gin,31,21,67.74
4,AU-How,13,11,84.62
5,AU-RDF,23,22,95.65
6,AU-Rig,41,20,48.78
7,AU-Stp,47,45,95.74
8,AU-TTE,30,29,96.67
9,AU-Wom,12,5,41.67
